# Lesson 03 - Mifumo ya Muundo ya Wakala

Katika somo hili, tunachunguza mifumo mitatu ya msingi ya muundo kwa ajili ya kujenga wawakilishi wa AI wenye ufanisi:

1. **Maelekezo Yenye Wazi kwa Wakala** — Kuunda maagizo sahihi, yanayobainisha majukumu yanayowaongoza wawakilishi
2. **Matokeo Yenye Muundo kwa Modeli za Pydantic** — Kuhakikisha wawakilishi wanarejesha data inayotarajiwa na kuthibitishwa
3. **Wawakilishi wa Majukumu Moja** — Kubuni wawakilishi waliolenga ambao kila mmoja hufanya jambo moja vizuri

Tutatumia kila mfumo kwa hali ya **mpendekeza wa maeneo ya kusafiria**, tukijenga polepole mfumo unaoweza kupendekeza maeneo, kuangalia upatikanaji, na kushughulikia mipangilio.


## Usanidi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Mfano 1: Maelekezo Dhahiri kwa Wakala

Mfano wenye athari kubwa pia ni rahisi: kuandika maelekezo wazi, yenye undani kwa wakala wako.

Maelekezo mazuri hufafanua:
- **Nani** wakala ni (tabia na sauti)
- **Nini** anapaswa kufanya (majukumu hatua kwa hatua)
- **Jinsi** anapaswa kujiendesha (vikwazo na mtindo)

Hapo chini, tunaunda wakala wa mshauri wa usafiri mwenye maelekezo ya wazi yanayoathiri kila jibu analotengeneza.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Mfano 2: Matokeo Yenye Muundo Kwa Modeli za Pydantic

Maandishi yasiyo na muundo ni muhimu kwa mazungumzo, lakini mifumo inayofuata inahitaji data yenye muundo.
Kwa kuunganisha **modeli za Pydantic** na **kifunction cha zana**, tunaweza:

- Kuweka muundo sahihi wa matokeo ya wakala
- Kuhakiki majibu kiotomatiki
- Kuunganisha matokeo ya wakala katika mantiki ya programu kwa uhakika

Pia tunatambulisha zana inayorejesha maelezo ya mahali pa kwenda ili wakala aanze mapendekezo yake kwa data halisi.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Mfano 3: Wakala wa Wajibu Mmoja

Kazi ngumu zinanufaika kwa kugawanya kazi kwa mawakala wengi wenye lengo maalum, kila mmoja akiwa na jukumu moja:

- **Mtaalamu wa Ukurasa** ambaye anajua kuhusu maeneo na upatikanaji
- **Mpangaji wa Usafirishaji** anayeshughulikia ndege, hoteli, na ratiba

Hii inaiga kanuni ya uhandisi wa programu ya *kutenganishwa kwa masuala* — kila wakala ni rahisi kupimwa, kudumishwa, na kuboreshwa kwa uhuru.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Muhtasari

Katika somo hili tulitumia mifumo mitatu ya muundo wa maajenti katika hali ya mshauri wa usafiri:

| Mfumo | Wazo Kikuu | Faida |
|---|---|---|
| **Maelekezo Wazi** | Tambua persona, majukumu, na vizingiti mapema | Tabia ya maajenti inayolingana na chapa |
| **Matokeo Yaliyopangwa** | Tumia mifano ya Pydantic kama muundo wa jibu | Matokeo yaliyothibitishwa, yanayoweza kusomwa na mashine |
| **Wajibu Mmoja** | Mpa kila maajenti kazi moja maalum | Rahisi kujaribu, kudumisha, na kuunganisha |

Mifumo hii huunganishwa kwa asili — unaweza kuchanganya maelekezo wazi na matokeo yaliyopangwa ndani ya maajenti wa wajibu mmoja kujenga mifumo imara, tayari kwa uzalishaji.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kisahihi**:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya utafsiri wa AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kwa usahihi, tafadhali fahamu kuwa tafsiri za kiotomatiki zinaweza kuwa na makosa au kasoro. Nyaraka asilia kwa lugha yake ya asili inapaswa kuzingatiwa kama chanzo cha mamlaka. Kwa habari muhimu, tafsiri ya kitaalamu ya binadamu inashauriwa. Hatutawajibika kwa kutoelewana au tafsiri potofu zinazotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
